# Module 1 — Customer Segmentation
**Dataset:** Online Retail II UCI — real e-commerce transactions (2010–2011)

**Techniques:** RFM Feature Engineering · K-Means · DBSCAN · Silhouette Score · Elbow Method · PCA

**Business Goal:** Segment customers into distinct behavioral groups to enable personalised marketing strategies.

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

from data_loader import load_raw, build_rfm

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
print('Libraries loaded.')

## 1. Load & Inspect Data

> **Setup:** Download `online_retail_II.xlsx` from https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci  
> Place it in `../data/raw/online_retail_II.xlsx` then run all cells.

In [ ]:
df = load_raw(sheet_name='Year 2010-2011')

print(f'Shape: {df.shape}')
print(f'Date range: {df["InvoiceDate"].min().date()} → {df["InvoiceDate"].max().date()}')
print(f'Unique customers: {df["CustomerID"].nunique():,}')
print(f'Unique products:  {df["StockCode"].nunique():,}')
print(f'Countries:        {df["Country"].nunique()}')
print(f'\nTop 5 countries:')
print(df['Country'].value_counts().head())
df.head()

## 2. RFM Feature Engineering

In [ ]:
rfm = build_rfm(df)
print(f'RFM table: {rfm.shape[0]:,} customers')
print(rfm[['Recency','Frequency','Monetary','AvgOrderValue','Tenure']].describe().round(2))
rfm.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes,
                           ['Recency','Frequency','Monetary'],
                           ['#3b82d4','#27ae60','#f39c12']):
    rfm[col].clip(upper=rfm[col].quantile(0.99)).hist(bins=60, ax=ax, color=color, edgecolor='white')
    ax.set_title(f'{col} Distribution (99th pct clip)', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Customers')
plt.tight_layout()
plt.savefig('../reports/rfm_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Scaling (Log + StandardScaler)

In [ ]:
CLUSTER_FEATURES = ['Recency', 'Frequency', 'Monetary']

rfm_log = rfm[CLUSTER_FEATURES].copy()
rfm_log['Frequency'] = np.log1p(rfm_log['Frequency'])
rfm_log['Monetary']  = np.log1p(rfm_log['Monetary'])

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)
print('Scaled shape:', rfm_scaled.shape)
print('Mean ~0, Std ~1 after scaling:', rfm_scaled.mean(axis=0).round(3), rfm_scaled.std(axis=0).round(3))

## 4. K-Means: Elbow + Silhouette

In [ ]:
inertia_vals = []
sil_scores   = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbls = km.fit_predict(rfm_scaled)
    inertia_vals.append(km.inertia_)
    sil_scores.append(silhouette_score(rfm_scaled, lbls))

optimal_k = list(K_range)[np.argmax(sil_scores)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(list(K_range), inertia_vals, 'bo-', lw=2, ms=8)
ax1.set_title('Elbow Method — Inertia vs K', fontweight='bold')
ax1.set_xlabel('k'); ax1.set_ylabel('Inertia')

ax2.plot(list(K_range), sil_scores, 'rs-', lw=2, ms=8)
ax2.axvline(optimal_k, color='green', linestyle='--', label=f'Optimal k={optimal_k}')
ax2.set_title('Silhouette Score vs K', fontweight='bold')
ax2.set_xlabel('k'); ax2.set_ylabel('Silhouette Score')
ax2.legend()

plt.tight_layout()
plt.savefig('../reports/kmeans_selection.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Optimal k = {optimal_k}  |  Silhouette = {max(sil_scores):.4f}')

## 5. Final K-Means Segmentation

In [ ]:
km_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
rfm['KMeans_Cluster'] = km_final.fit_predict(rfm_scaled)

profile = rfm.groupby('KMeans_Cluster').agg(
    CustomerCount = ('CustomerID', 'count'),
    Avg_Recency   = ('Recency',    'mean'),
    Avg_Frequency = ('Frequency',  'mean'),
    Avg_Monetary  = ('Monetary',   'mean'),
    Avg_AOV       = ('AvgOrderValue','mean')
).round(1)

# Business labels — assign based on ranked RFM profile
LABELS = {0:'Champions', 1:'Loyal Customers', 2:'At-Risk', 3:'New / Occasional', 4:'Potential Loyalists'}
mon_rank = profile['Avg_Monetary'].rank(ascending=False).astype(int)
rec_rank = profile['Avg_Recency'].rank(ascending=True).astype(int)
combined = (mon_rank + rec_rank).rank().astype(int) - 1
label_keys = sorted(LABELS.keys())[:len(profile)]
profile['Segment_Label'] = combined.map({i: LABELS[k] for i, k in enumerate(label_keys)})

cluster_to_label = profile['Segment_Label'].to_dict()
rfm['Segment'] = rfm['KMeans_Cluster'].map(cluster_to_label)

print(profile)
print(f'\nTotal customers: {len(rfm):,}')

In [ ]:
# Segment size + RFM heatmap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

seg_counts = rfm['Segment'].value_counts()
colors = ['#3b82d4','#27ae60','#f39c12','#e74c3c','#9b59b6']
ax1.bar(seg_counts.index, seg_counts.values, color=colors[:len(seg_counts)], edgecolor='white')
for i, (lbl, val) in enumerate(seg_counts.items()):
    ax1.text(i, val + 5, str(val), ha='center', fontweight='bold')
ax1.set_title('Customers per Segment', fontweight='bold')
ax1.set_ylabel('Count')
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=20, ha='right')

heatmap_data = profile[['Avg_Recency','Avg_Frequency','Avg_Monetary']].copy()
heatmap_norm = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min())
heatmap_norm.index = profile['Segment_Label']
sns.heatmap(heatmap_norm, annot=heatmap_data.values, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax2, annot_kws={'size':10})
ax2.set_title('Segment RFM Heatmap (normalised colour, raw value)', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/segment_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. PCA Visualisation

In [ ]:
pca = PCA(n_components=2, random_state=42)
rfm_pca = pca.fit_transform(rfm_scaled)

plt.figure(figsize=(10, 7))
for i, (seg, grp) in enumerate(rfm.groupby('Segment')):
    idx = grp.index
    plt.scatter(rfm_pca[idx, 0], rfm_pca[idx, 1],
                label=seg, alpha=0.5, s=20, color=colors[i % len(colors)])
plt.title(f'K-Means Segments (k={optimal_k}) — PCA Projection', fontsize=14, fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend(markerscale=2)
plt.tight_layout()
plt.savefig('../reports/kmeans_pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. DBSCAN — Density-Based Outlier Detection

In [ ]:
# k-distance graph to choose eps
nbrs = NearestNeighbors(n_neighbors=5).fit(rfm_scaled)
dists, _ = nbrs.kneighbors(rfm_scaled)
dists = np.sort(dists[:, 4])

plt.figure(figsize=(8, 4))
plt.plot(dists)
plt.title('k-Distance Graph — DBSCAN eps Selection', fontweight='bold')
plt.xlabel('Points (sorted)'); plt.ylabel('5th-NN Distance')
plt.tight_layout()
plt.savefig('../reports/dbscan_eps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=5)
rfm['DBSCAN_Cluster'] = dbscan.fit_predict(rfm_scaled)

n_clusters_db = len(set(rfm['DBSCAN_Cluster'])) - (1 if -1 in rfm['DBSCAN_Cluster'].values else 0)
n_outliers    = (rfm['DBSCAN_Cluster'] == -1).sum()
pct           = n_outliers / len(rfm) * 100
print(f'DBSCAN clusters: {n_clusters_db}')
print(f'Outliers (noise): {n_outliers} ({pct:.1f}%)')

# Profile the outliers — often VIP or erratic buyers
outliers = rfm[rfm['DBSCAN_Cluster'] == -1]
print('\nOutlier RFM profile:')
print(outliers[['Recency','Frequency','Monetary']].describe().round(1))

## 8. Save Outputs

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

rfm.to_csv('../data/processed/rfm_segments.csv', index=False)
profile.to_csv('../data/processed/segment_profiles.csv')

print('Saved: rfm_segments.csv, segment_profiles.csv')
print('\n=== Module 1 Complete ===')
print(f'Customers segmented : {len(rfm):,}')
print(f'K-Means segments    : {optimal_k}')
print(f'Silhouette score    : {max(sil_scores):.4f}')
print(f'DBSCAN outliers     : {n_outliers} ({pct:.1f}%)')
print(f'\nSegment breakdown:')
print(rfm['Segment'].value_counts())